# Optimization

## import some modules

In [1]:
from dftpy.ions import Ions
from dftpy.field import DirectField
from dftpy.grid import DirectGrid
from dftpy.functional import LocalPseudo, Functional, TotalFunctional
from dftpy.formats import io
from dftpy.math_utils import ecut2nr
from dftpy.time_data import TimeData
from dftpy.optimization import Optimization
from dftpy.mpi import sprint

## pseudopotential file

In [21]:
path_pp='../DATA/'
file1='al.lda.recpot'
PP_list = {'Al': path_pp+file1}

## build the ions or read from file

In [22]:
from ase.build import bulk
atoms = bulk('Al', 'fcc', a=4.05, cubic=True)
ions = Ions.from_ase(atoms)
# ions = io.read(posfile)

In [23]:
len(ions)

4

## make a grid

In [30]:
nr = ecut2nr(ecut=35, lattice=ions.cell)
grid = DirectGrid(lattice=ions.cell, nr=nr)
sprint('The final grid size is ', nr)

The final grid size is  [20 20 20]


##  build local pseudo, and generate guess density

In [31]:
PSEUDO = LocalPseudo(grid = grid, ions=ions, PP_list=PP_list)

rho_ini = DirectField(grid=grid)
rho_ini[:] = ions.get_ncharges()/ions.cell.volume

setting key: Al -> ../DATA/al.lda.recpot


## instance KEDF, XC and HARTREE functionals

In [32]:
KE = Functional(type='KEDF',name='TFvW')
XC = Functional(type='XC',name='LDA')
HARTREE = Functional(type='HARTREE')

## instance DFTpy evaluator

In [33]:
evaluator = TotalFunctional(KE=KE, XC=XC, HARTREE=HARTREE, PSEUDO=PSEUDO)

## instance and execute DFTpy density optimizer

In [34]:
optimization_options = {'econv' : 1e-6*ions.nat}
opt = Optimization(EnergyEvaluator=evaluator, optimization_options = optimization_options,
        optimization_method = 'TN')

rho = opt.optimize_rho(guess_rho=rho_ini)

Step    Energy(a.u.)            dE              dP              Nd      Nls     Time(s)         
0       -7.465944503246E+00     -7.465945E+00   1.195655E+00    1       1       2.717805E-02    
1       -7.709226052702E+00     -2.432815E-01   4.021914E-02    2       2       4.101396E-02    
2       -7.715744270628E+00     -6.518218E-03   2.489284E-03    6       3       6.110215E-02    
3       -7.715925713678E+00     -1.814431E-04   1.772537E-04    6       3       7.637501E-02    
4       -7.715938122302E+00     -1.240862E-05   1.486108E-05    7       2       8.934307E-02    
5       -7.715939014877E+00     -8.925752E-07   9.044195E-07    6       2       1.015720E-01    
6       -7.715939068005E+00     -5.312726E-08   7.416794E-08    5       2       1.121180E-01    
#### Density Optimization Converged ####
Chemical potential (a.u.): 0.2874858982183081
Chemical potential (eV)  : 7.822889752980333


## evaluate final energy

In [37]:
energy = evaluator.Energy(rho=rho, ions=ions)
print('Energy (eV/atom)', energy/4*27.211385)

Energy (eV/atom) -52.49034715400295


##  print the timing

In [10]:
TimeData.output(lprint=True, sort='cost')

--------------------------------Time information--------------------------------
Label                       Cost(s)                 Number          Avg. Cost(s)            
ewald.Energy_corr           0.0000                  1               0.0000                  
CBspline._calc_PME_Qarray   0.0020                  1               0.0020                  
ewald.Energy_rec_PME        0.0030                  1               0.0030                  
ewald.Energy_real_fast2     0.0073                  1               0.0073                  
LocalPseudo.local_PP        0.0074                  1               0.0074                  
TF                          0.0079                  52              0.0002                  
ewald.energy                0.0103                  52              0.0002                  
LDA                         0.0150                  52              0.0003                  
XC.compute                  0.0155                  52              0.0003        

In [11]:
rho.write('rho.xsf', ions=ions)
rho.write('rho.cube', ions=ions)

## Visualize with scikit-image and matplotlib

!pip install scikit-image matplotlib

In [12]:
from dftpy.visualize import view

## Visualize with VESTA

In [13]:
view(ions=ions, data=rho, viewer='vesta')

save ./.dftpy.xsf in darwin platform


In [38]:
rho

DirectField([[[0.0061812 , 0.00804475, 0.01416318, 0.02314991,
               0.02917551, 0.02930223, 0.02662973, 0.02407038,
               0.02231122, 0.02129988, 0.02097181, 0.02129988,
               0.02231123, 0.02407038, 0.02662973, 0.02930223,
               0.02917551, 0.02314991, 0.01416318, 0.00804475],
              [0.00804475, 0.01003921, 0.0162089 , 0.0245521 ,
               0.02959833, 0.02926775, 0.02662199, 0.02419961,
               0.02254569, 0.02160205, 0.02129988, 0.02160205,
               0.02254569, 0.02419961, 0.02662199, 0.02926775,
               0.02959833, 0.0245521 , 0.0162089 , 0.01003921],
              [0.01416318, 0.0162089 , 0.0217219 , 0.02775689,
               0.03031732, 0.02908728, 0.02665877, 0.02464721,
               0.023299  , 0.02254569, 0.02231122, 0.02254569,
               0.023299  , 0.02464722, 0.02665877, 0.02908728,
               0.03031733, 0.02775689, 0.0217219 , 0.0162089 ],
              [0.02314991, 0.0245521 , 0.02775689, 0

In [40]:
rhog = rho.fft()

In [41]:
rhog[0,0,0]

(11.999999999999991+0j)